# Lab command cookbook

The commands of this lab that are hard to remember or to type: `kubectl` with jsonpath and jq,
`argocd`, `helm`, kro, ACK and Moto, Kyverno, Prometheus and Loki. Each section starts with the
concept in a few lines, then cells you can run as they are.

**Every cell only reads.** Running the whole notebook never changes the lab. Commands that change
something are shown as plain text in *Changes go through Git* at the end, with the Git-first way
to do the same thing. In this lab, Argo CD reverts most direct `kubectl` changes within minutes
(self-heal), so the durable change is always a commit.

**Setup (once):** `bash scripts/setup-notebook.sh`, then in VS Code
*Select Kernel → Jupyter Kernel… → Bash*. Details: [README.md](README.md).

**Secrets:** no cell prints a password, token or Secret value, and saved outputs are stripped
before a commit. Keep it that way when you add cells: `make password` belongs in the terminal.

Related reading: [Concepts and glossary](../../concepts-and-glossary.md),
[Lab 0 guided tour](../../lab-0-guided-tour.md), [runbooks](../../runbooks/).

## 0. Setup and tool check

Defines the variables the other cells use. Run it first, and again after restarting the kernel.

In [ ]:
HUB=k3d-hub-cluster
NONPROD=k3d-spoke-nonprod
PROD=k3d-spoke-prod
SPOKES=("$NONPROD" "$PROD")
SECRET_DIR="${GITOPS_LAB_SECRET_DIR:-$HOME/.config/gitops-lab}"
LAB_REPO="$(git rev-parse --show-toplevel 2>/dev/null || echo "$HOME/repos/gitops-control-plane")"
CATALOG_REPO="$(dirname "$LAB_REPO")/platform-catalog"
MOTO_ENDPOINT=http://localhost:5000

for t in kubectl argocd helm aws jq yq k3d docker; do
  if command -v "$t" >/dev/null; then echo "✔ $t"; else echo "✘ $t missing"; fi
done
for c in "$HUB" "$NONPROD" "$PROD"; do
  if kubectl config get-contexts -o name | grep -qx "$c"; then echo "✔ context $c"; else echo "✘ context $c missing"; fi
done
echo "lab repo: $LAB_REPO"
echo "catalog:  $CATALOG_REPO"

**Reading the output.** Every line should start with ✔.

- **✘ next to a tool:** the cells that use it fail with `command not found`. Install it on the lab
  host, run `make notebook-setup` again (the kernel's PATH is fixed at setup) and restart the kernel.
- **✘ next to a context:** that cluster is not in your kubeconfig. Is the lab running (`make start`)?
- **The two paths** are where later cells read `clusters/blueprint-revisions.env` and the catalog's
  values files.

## 1. Where am I?

Three k3d clusters: the **hub** runs Argo CD, Keycloak, Grafana, Prometheus and Loki; the two
**spokes** run the platform add-ons (kro, ACK, Kyverno, …) and the tenant apps. Every command
names its cluster with `--context`, so it never depends on your current context.

In [ ]:
for c in "$HUB" "${SPOKES[@]}"; do
  echo "== $c"
  kubectl --context "$c" get nodes -o custom-columns='NODE:.metadata.name,READY:.status.conditions[?(@.type=="Ready")].status,VERSION:.status.nodeInfo.kubeletVersion,IP:.status.addresses[?(@.type=="InternalIP")].address'
done

**Reading the output.** Each node is a Docker container that k3d runs
(`docker ps` lists them).

- **`server-0`** runs the control plane (API server, scheduler, etcd/SQLite) and workloads too; the
  spokes add **`agent-0`**, a worker.
- **`READY True`** means the node's kubelet reports healthy. `False` or `Unknown` usually means the
  container stopped: check `docker ps`, then `make start`.
- **`VERSION`** is the k3s build of Kubernetes; all clusters run the same one.
- **`IP`** is the node's address on the Docker network `k3d-cloud-net`, which Moto shares. That is
  how Argo CD on the hub reaches the spoke API servers (`https://k3d-spoke-nonprod-server-0:6443`) and
  the ACK controllers reach Moto (`http://moto-cloud:5000`).

Every Argo CD Application with its sync and health status, the one-line answer to *"is the lab
green?"*. Sync = does the cluster match Git; health = is what runs actually working.

In [ ]:
kubectl --context "$HUB" -n argocd get applications \
  -o custom-columns='APP:.metadata.name,PROJECT:.spec.project,SYNC:.status.sync.status,HEALTH:.status.health.status'

**Reading the output.** The two status columns answer different
questions: *is the cluster what Git says?* and *does it work?*

| SYNC | Meaning |
| --- | --- |
| `Synced` | the live objects match what Git renders |
| `OutOfSync` | they differ: a commit not applied yet (automated sync applies it within about 3 minutes) or a manual change (self-heal reverts it) |
| `Unknown` | Argo CD could not render or compare, e.g. an unreachable repo or a chart error |

| HEALTH | Meaning |
| --- | --- |
| `Healthy` | every resource reports healthy |
| `Progressing` | a rollout or sync is still running; normal for a minute or two |
| `Degraded` | something failed: a crash-looping Pod, a failed Job, a resource with an error condition |
| `Missing` | Git says it should exist, but it does not |

**Synced + Degraded:** Git was applied, but the app does not work; look at its Pods.
**OutOfSync + Healthy:** it works, but it is not what Git says. `PROJECT` is the AppProject that limits
where the app may deploy.

Only the problems (empty output means everything is Synced and Healthy):

In [ ]:
kubectl --context "$HUB" -n argocd get applications -o json | jq -r '
  .items[]
  | select(.status.sync.status != "Synced" or .status.health.status != "Healthy")
  | "\(.metadata.name)\t\(.status.sync.status)\t\(.status.health.status)\t\(.status.conditions[0].message // "")"'
echo "(end of list)"

**Reading the output.** Only
`(end of list)` means every app is Synced and Healthy. Otherwise each line shows the app, its sync and
health status, and its first condition message, usually the quickest clue (a `ComparisonError`
means Argo CD could not render the app from Git). Next step: `app get` in section 2.

## 2. Argo CD

- An **Application** points at one place in Git (a path, or a Helm chart plus values) and one
	destination cluster and namespace.
- An **ApplicationSet** generates Applications, here one per spoke from the cluster Secrets and
	their labels (`environment`) and annotations (`blueprints-revision`).
- An **AppProject** limits which repos, clusters, namespaces and kinds an Application may use.

The CLI below logs in as `platform-admin` with a **temporary config file**, so it never touches
your own `argocd` login. The password is read from its file and never shown.

In [ ]:
ARGOCD_CFG="$(mktemp)"
argocd login localhost --plaintext --grpc-web --skip-test-tls --config "$ARGOCD_CFG" \
  --username platform-admin --password "$(cat "$SECRET_DIR/argocd-platform-admin.password")" \
  </dev/null >/dev/null && echo "✔ logged in (temporary config, removed in the last cell)"
A=(argocd --config "$ARGOCD_CFG")

In [ ]:
"${A[@]}" app list

**Reading the output.** The same apps through the Argo CD API, with more columns (scroll
sideways):

- **`CLUSTER`**: the destination API server. `https://kubernetes.default.svc` is the hub itself.
- **`SYNCPOLICY`**: `Auto` applies every Git change; `Auto-Prune` also deletes resources removed from
  Git (without prune they stay, marked OutOfSync).
- **`CONDITIONS`**: warnings and errors, such as `SharedResourceWarning` (two apps manage the same
  object). `<none>` is what you want.
- **`REPO`**: where the desired state comes from: a Git repo, or a Helm repository or OCI registry.

One app in detail: source, destination, conditions and every resource it manages.

In [ ]:
APP=orders-dev
"${A[@]}" app get "$APP"

**Reading the output.** `orders-dev` has **two sources**:

1. the Helm chart from `ghcr.io/brunobml/charts`, pinned to `1.0.0` (owned by the platform);
2. the app repo `orders-processor` on `main`, marked `Ref: values`: its `deploy/values-dev.yaml`
   feeds the chart through `$values/...` (owned by the team).

This *multi-source* pattern versions the chart and the values separately. `Sync Status: Synced to
1.0.0` names the revision of the first source; `Server: spoke-nonprod` is the destination cluster.
Further down, every managed object is listed with its own sync and health status: that table tells
you *which* object makes an app OutOfSync or Degraded.

Deployment history (which Git revision was synced when) and the live diff against Git. `app diff` exits with 1 when there is a difference, so the cell prints the code instead of failing.

In [ ]:
"${A[@]}" app history "$APP"
echo
"${A[@]}" app diff "$APP"; rc=$?
echo "app diff exit code: $rc (0 = live matches Git, 1 = differences shown above)"

**Reading the output.** 

- **History:** one row per sync that changed what is deployed, per source (this app has two). `ID`
  is what `argocd app rollback <app> <ID>` takes; a rollback is a direct change and needs automated
  sync off, so in this lab you revert in Git instead.
- **Diff:** no output and exit code 0 means the live objects equal what Git renders. A difference
  prints as a unified diff, `-` live and `+` desired. Typical causes: a manual `kubectl edit`
  (self-heal reverts it) or a controller that writes into the object (that needs `ignoreDifferences`
  on the app).

Which ApplicationSet generated which Application:

In [ ]:
kubectl --context "$HUB" -n argocd get applications -o json | jq -r '
  .items[]
  | select(.metadata.ownerReferences[]?.kind == "ApplicationSet")
  | "\(.metadata.ownerReferences[0].name)\t\(.metadata.name)"' | sort | column -t

**Reading the output.** Left, the ApplicationSet;
right, the Applications it generated.

- **Most spoke add-ons appear twice** (`-spoke-nonprod`, `-spoke-prod`): the *cluster generator*
  creates one app per cluster Secret that matches its selector.
- **`addon-ack-s3` exists only for nonprod**: its ApplicationSet selects `environment: nonprod`.
- **Apps missing here** (`root-control-plane`, `argo-cd`, `platform-projects`, the hub add-ons) are
  plain Applications kept in Git and applied by the root app: the *app of apps* pattern.
- **Deleting a generated app by hand does not stick**: the ApplicationSet recreates it. Change the
  ApplicationSet or the cluster labels in Git instead.

**Promotion state.** spoke-nonprod follows `main` of platform-catalog, spoke-prod a pinned tag.
The value lives on the cluster Secret (labels and annotations only are shown, never its data) and
comes from `clusters/blueprint-revisions.env` via `make promote-blueprints`.

In [ ]:
kubectl --context "$HUB" -n argocd get secrets -l argocd.argoproj.io/secret-type=cluster \
  -o custom-columns='CLUSTER:.metadata.name,ENV:.metadata.labels.environment,BLUEPRINTS:.metadata.annotations.blueprints-revision'
echo
grep -v '^#' "$LAB_REPO/clusters/blueprint-revisions.env"

**Reading the output.** Both views must agree.

- **`main`**: spoke-nonprod gets every platform-catalog commit as soon as it is pushed, so blueprints
  and controller values are tried there first.
- **`v1.16.0`**: spoke-prod changes only when a tag is promoted.
- **They disagree:** someone edited `blueprint-revisions.env` without running
  `make promote-blueprints`, or the reverse.

## 3. Helm, and why `helm list` is (almost) empty

Argo CD renders Helm charts itself (`helm template`) and applies the result, so there are no Helm
*releases* for the charts it manages. On the spokes `helm list` shows only what k3s installs on
its own (Traefik). To see what Argo CD would apply, render the chart locally with the same values.

In [ ]:
helm --kube-context "$NONPROD" list -A

**Reading the output.** Only Traefik shows up, installed by k3s's
built-in Helm controller (the `helm-install-traefik` Job). None of the charts Argo CD deploys is here,
by design: Argo CD runs `helm template` and applies the result, so no Helm release exists. That is
why `helm upgrade`, `helm rollback` and `helm uninstall` are not the tools for this lab's add-ons:
Git is.

Every Helm chart Argo CD deploys, with its pinned version:

In [ ]:
kubectl --context "$HUB" -n argocd get applications -o json | jq -r '
  .items[] | .metadata.name as $app
  | (.spec.sources // [.spec.source])[]
  | select(.chart)
  | "\($app)\t\(.chart)\t\(.targetRevision)\t\(.repoURL)"' | sort | column -t

**Reading the output.** One line per app and chart: app, chart, pinned chart version, chart
source.

- **Sources:** `https://...` are classic Helm repositories; `public.ecr.aws/...`,
  `registry.k8s.io/...` and `ghcr.io/...` are OCI registries.
- **Versions:** controller charts are pinned in this repo's ApplicationSets, so both spokes run the
  same chart version. Their *values* come from platform-catalog at each spoke's revision, which is
  how a values change reaches nonprod before prod.
- **Upgrading a chart:** change `targetRevision` in the ApplicationSet, render it first (next
  cells), then commit.

A chart's metadata and the first default values (HTTP repo and OCI registry):

In [ ]:
helm show chart kyverno --repo https://kyverno.github.io/kyverno/ --version 3.9.1 | grep --color=never -E '^(name|version|appVersion):'
echo
helm show chart oci://public.ecr.aws/aws-controllers-k8s/s3-chart --version 1.13.0 2>/dev/null | grep --color=never -E '^(name|version|appVersion):'
echo
helm show values kyverno --repo https://kyverno.github.io/kyverno/ --version 3.9.1 | head -20

**Reading the output.** 

- **`version`** is the chart's own version, the one Argo CD pins (`3.9.1`); **`appVersion`** is the
  software inside (Kyverno `v1.19.1`). They move independently: a chart release can change only
  templates or defaults.
- **The values listing** shows the chart's defaults. The lab overrides a few of them in
  `platform-catalog/controllers/kyverno/values-kyverno.yaml`; everything else keeps the default.

Render Kyverno the way Argo CD does for the spokes: the catalog values file plus the parameter
from `applicationsets/addons-spoke-kyverno.yaml`, then list the kinds and names it would create.
The catalog checkout should match the spoke's revision (`main` for nonprod).

In [ ]:
helm template kyverno kyverno --repo https://kyverno.github.io/kyverno/ --version 3.9.1 -n kyverno \
  -f "$CATALOG_REPO/controllers/kyverno/values-kyverno.yaml" \
  --set reportsController.enabled=true \
  | yq -N '.kind + "/" + .metadata.name' | sort | uniq -c | sort -rn | head -25

**Reading the output.** Every object the chart renders with the lab's values,
counted by kind and name: what Argo CD would apply. Render, change a value, render again and compare
before you commit a values change; `helm template` touches no cluster. The Jobs and some Pods are
Helm hooks (`helm.sh/hook` annotation), which Argo CD runs during a sync instead of keeping, so not
every line becomes a lasting object.

## 4. kro: platform APIs

A **ResourceGraphDefinition** (RGD) defines a new Kubernetes kind (`QueueBackedService`,
`TestReportViewer`, …) and the resources one instance of it creates. A team writes a small
instance (a *claim*); kro creates and wires the children and reports their state back.

In [ ]:
kubectl --context "$NONPROD" get rgd

**Reading the output.** One row per platform API.

- **`KIND` and `APIVERSION`** are what a claim uses: `apiVersion: kro.run/v1alpha1`,
  `kind: QueueBackedService`.
- **`STATE Active`**: kro created the CRD for that kind and runs a controller for it.
- **`READY False`**: the RGD itself is broken: an invalid CEL expression, an unknown kind in a
  template, or missing RBAC (kro gets permissions per RGD through an aggregated ClusterRole). Its
  conditions say which: `kubectl --context "$NONPROD" get rgd <name> -o yaml`.

The fields a claim may set (kro generates the CRD, so `kubectl explain` works):

In [ ]:
kubectl --context "$NONPROD" explain queuebackedservice.spec

**Reading the output.** The fields a claim may set and their types. kro
generated this schema from the RGD's `spec.schema`, so the API server validates claims like any other
resource: a wrong type is rejected when the claim is applied, before kro sees it. `<no description>`
only means the RGD gives none. Drill down with `kubectl explain queuebackedservice.spec.replicas`.

All instances on both spokes:

In [ ]:
for c in "${SPOKES[@]}"; do
  echo "== $c"
  kinds=$(kubectl --context "$c" get rgd -o jsonpath='{.items[*].spec.schema.kind}' | tr ' ' ',')
  kubectl --context "$c" get "$kinds" -A
done

**Reading the output.** One row per instance (claim) of every kind the spoke's RGDs define.

- **`ACTIVE` + `READY True`**: kro created every child and they meet their readiness rules.
- **`IN_PROGRESS`**: kro is still creating children or waiting for one, e.g. an ACK resource waiting
  for the AWS side.
- **`ERROR` + `READY False`**: kro could not apply a child. The conditions name the object and the
  API error. When this cell was written, both `teamekscluster` instances were in this state: their
  `ResourcesReady` condition said the EKS `Nodegroup` was rejected because `spec.subnets` cannot change
  once set. To read them, change `NS`, `NAME` and the kind in the conditions cell below.

Everything one instance created, found through the `kro.run/instance-*` labels kro puts on its children:

In [ ]:
NS=orders-dev; NAME=orders
kubectl --context "$NONPROD" get deploy,svc,ingress,serviceaccount,configmap,queues.sqs.services.k8s.aws -n "$NS" \
  -l "kro.run/instance-name=$NAME" -o custom-columns='KIND:.kind,NAME:.metadata.name,NODE:.metadata.labels.kro\.run/node-id'

**Reading the NODE column.** In kro a *node* is one resource of the RGD's dependency graph (a DAG),
not a Kubernetes worker node (`kubectl get nodes`).

- Each entry of the RGD's `spec.resources` has an `id`; that id is the node:

  ```yaml
  resources:
    - id: dlq      # kro.run/node-id: dlq
      template:
        kind: Queue
        ...
    - id: queue    # kro.run/node-id: queue
      template:
        kind: Queue
        ...        # redrivePolicy uses ${dlq.status.ackResourceMetadata.arn}
  ```

- References between nodes (CEL expressions such as `${dlq.status...}`) are the edges of the graph
  and set the order: kro creates `dlq` before `queue`, which needs its ARN, and fills the ConfigMap
  `config` (read by the Deployment) once `${queue.status.queueURL}` exists.
- kro labels every child it creates with `kro.run/node-id: <id>`; the cell above uses these labels.

In [ ]:
kubectl --context "$NONPROD" -n "$NS" get queuebackedservice "$NAME" -o json \
  | jq -r '.status.conditions[]? | "\(.type)\t\(.status)\t\(.reason // "")\t\(.message // "")"' | column -t -s $'\t'

**Reading the conditions.** Controllers report progress as conditions (`Type`, `Status`,
`Reason`, `Message`). kro sets four on every instance:

| Condition | `True` means |
| --- | --- |
| `InstanceManaged` | kro manages the instance: its finalizer and labels are in place, so deleting the instance also removes its children. |
| `GraphResolved` | kro evaluated the RGD's CEL expressions for this instance: no missing or circular references. |
| `ResourcesReady` | every child exists and meets its readiness rule (below). |
| `Ready` | the summary of the three; check this one first. |

**Ready is not the same as healthy.** kro only checks the readiness rules an RGD declares
(`readyWhen`) and waits for status fields other nodes reference, such as `${queue.status.queueURL}`.
A child without a rule counts as ready once it exists. `queue-backed-service` declares no
`readyWhen`, so `Ready=True` does not prove the worker has its replicas running: check
`kubectl get deploy` or the app's health in Argo CD.

## 5. ACK and Moto: "AWS" resources

ACK controllers turn Kubernetes resources (`Queue`, `Bucket`, `Role`, `VPC`, …) into AWS API
calls. Here the AWS API is **Moto**, an in-memory emulator, so everything is local and lost when
Moto restarts (`make moto-restart` recreates it). Each namespace is mapped to an AWS account by
the annotation `services.k8s.aws/owner-account-id`.

In [ ]:
for c in "${SPOKES[@]}"; do
  echo "== $c"
  kubectl --context "$c" get ns -o custom-columns='NAMESPACE:.metadata.name,ACCOUNT:.metadata.annotations.services\.k8s\.aws/owner-account-id' \
    | awk 'NR==1 || $2!="<none>"'
done

**Reading the output.** Only namespaces with the annotation are listed.
ACK creates the AWS resources of each namespace in that account: nonprod namespaces map to
`111111111111`, prod namespaces to `222222222222`, so dev/test and prod live in separate accounts as in
a real AWS organization. The ACK controller switches account by assuming a role there (ACK's
cross-account resource management). The mapping sits on the namespace, so a team cannot pick another
account in its claim.

ACK resources with their account and conditions (`ACK.ResourceSynced=True` means the AWS side matches):

In [ ]:
for c in "${SPOKES[@]}"; do
  echo "== $c"
  kubectl --context "$c" get queues.sqs.services.k8s.aws,buckets.s3.services.k8s.aws,roles.iam.services.k8s.aws -A -o json 2>/dev/null | jq -r '
    .items[]
    | "\(.kind)\t\(.metadata.namespace)/\(.metadata.name)\t\(.status.ackResourceMetadata.ownerAccountID // "-")\t\([.status.conditions[]? | "\(.type)=\(.status)"] | join(","))"' \
    | column -t -s $'\t'
done

**Reading the output.** Kind,
namespace/name, the AWS account the resource lives in, and its conditions.

| Condition | Meaning |
| --- | --- |
| `ACK.ResourceSynced=True` | the AWS side matches the spec. `False`: ACK is still working, or AWS (here Moto) refused a call; the condition message says why |
| `ACK.LateInitialized=True` | ACK copied defaults that AWS chose (queue attributes, for example) back into the spec |
| `ACK.Terminal=True` | a permanent error: retrying will not help, the spec must change |
| `Ready=True` | ACK's summary |

A `-` in the account column means ACK never reconciled the resource; compare with the namespace
annotation above. Details: `kubectl get <kind> <name> -n <ns> -o yaml`.

Ask Moto directly, as one account. `moto_as` assumes the lab's `smoke-test` role in that account
and runs one `aws` command with the temporary credentials (they are never printed). Accounts:
`111111111111` spoke-nonprod, `222222222222` spoke-prod, `123456789012` Moto's default (should
stay empty). For the full inventory: `make list-moto-resources`.

In [ ]:
moto_as() {
  local account="$1"; shift
  local creds
  creds=$(AWS_ACCESS_KEY_ID=mock-key AWS_SECRET_ACCESS_KEY=mock-secret \
    aws --endpoint-url "$MOTO_ENDPOINT" --region us-east-1 --cli-connect-timeout 5 sts assume-role \
      --role-arn "arn:aws:iam::${account}:role/smoke-test" --role-session-name notebook \
      --query Credentials --output json) || return 1
  AWS_ACCESS_KEY_ID=$(jq -r .AccessKeyId <<<"$creds") AWS_SECRET_ACCESS_KEY=$(jq -r .SecretAccessKey <<<"$creds") \
    AWS_SESSION_TOKEN=$(jq -r .SessionToken <<<"$creds") \
    aws --endpoint-url "$MOTO_ENDPOINT" --region us-east-1 --cli-connect-timeout 5 "$@"
}
for acct in 111111111111 222222222222; do
  echo "== account $acct"
  moto_as "$acct" sqs list-queues --query 'QueueUrls' --output text
  moto_as "$acct" s3api list-buckets --query 'Buckets[].Name' --output text
done

**Reading the output.** The same resources, seen from the AWS side: queue URLs
(`http://localhost:5000/<account>/<name>`) and bucket names per account. Compare with the ACK cell
above to tell two kinds of drift apart:

- **In Kubernetes but not here:** Moto lost its state (it keeps everything in memory);
  `make moto-restart` restarts it and lets ACK recreate the resources.
- **Here but not in Kubernetes:** an orphan, created outside ACK or left behind after a delete.

Moto's default account `123456789012` should stay empty; anything there used the wrong credentials.

## 6. Kyverno and Policy Reporter

Kyverno checks resources at **admission** (when they are created or changed) and in the
**background** (existing resources, written as PolicyReports). `Deny` policies block; `Audit`
policies only report. The UI for the reports is http://policy-reporter.localhost
(see [the Policy Reporter runbook](../../runbooks/policy-reporter.md)).

In [ ]:
kubectl --context "$NONPROD" get validatingpolicies.policies.kyverno.io,imagevalidatingpolicies.policies.kyverno.io -o json | jq -r '
  .items[]
  | "\(.metadata.name)\t\(.spec.validationActions // ["Deny"] | join(","))\t\(.metadata.annotations["policies.kyverno.io/severity"] // "-")\t\(.metadata.annotations["policies.kyverno.io/title"] // "-")"' \
  | column -t -s $'\t'

**Reading the output.** Name, action,
severity and title of each policy (the last two from its annotations).

- **`Audit`** policies only report: violations show up in PolicyReports and never block.
- **`Deny`** blocks the request at admission. Only `tenant-images-signed` denies, and only in
  namespaces labelled `platform.lab/image-verification=enabled`.
- **Severity** drives the badge in Policy Reporter and tells you which failures to fix first.

Pass/fail per policy on each spoke, from the PolicyReports:

In [ ]:
for c in "${SPOKES[@]}"; do
  echo "== $c"
  { kubectl --context "$c" get policyreports.wgpolicyk8s.io -A -o json; kubectl --context "$c" get clusterpolicyreports.wgpolicyk8s.io -o json; } \
    | jq -rs '[.[].items[].results[]?] | group_by(.policy)[]
        | "\(.[0].policy)\tpass=\(map(select(.result=="pass"))|length)\tfail=\(map(select(.result=="fail"))|length)"' \
    | column -t -s $'\t'
done

**Reading the output.** For each policy, how many existing resources pass or fail it,
from Kyverno's background scan.

- **One workload usually counts twice:** Kyverno checks the Pods and the Deployments, DaemonSets,
  StatefulSets, Jobs and CronJobs that create them, so a failing Deployment also has a failing Pod.
- **`fail=0` on both spokes for a while:** the Audit policy is a candidate for Deny (see the Policy
  Reporter runbook).
- **`pass=0 fail=0`:** nothing matched; check the policy's selector and kinds.

Every failing resource, grouped by policy and namespace (the list to work through):

In [ ]:
kubectl --context "$NONPROD" get policyreports.wgpolicyk8s.io -A -o json | jq -r '
  .items[] | .metadata.namespace as $ns | .scope as $s
  | .results[]? | select(.result == "fail")
  | "\(.policy)\t\($ns)\t\($s.kind)/\($s.name)"' | sort | column -t -s $'\t'

**Reading the output.** The work list: policy, namespace, failing object. Fix
the template (the Deployment), not the Pod: Pods are recreated from it. The result flips at the next
admission or background scan; results for objects Kyverno no longer evaluates can take about an hour
to disappear (Policy Reporter runbook, troubleshooting).

**Why would this be denied?** `--dry-run=server` sends a request through all admission checks
(Pod Security, ValidatingAdmissionPolicies, Kyverno) without creating anything. The two requests
below go to the `admission-probes` namespace, which exists for exactly this. Both are expected to
be denied: the first by the registry allowlist (a ValidatingAdmissionPolicy), the second by the
Kyverno image policy `tenant-images-signed`. The cell prints each reason.

In [ ]:
POD_SPEC='{"spec":{"securityContext":{"runAsNonRoot":true,"runAsUser":1000,"seccompProfile":{"type":"RuntimeDefault"}},"containers":[{"name":"probe","image":"IMAGE","securityContext":{"allowPrivilegeEscalation":false,"capabilities":{"drop":["ALL"]}}}]}}'
try_pod() {
  kubectl --context "$NONPROD" -n admission-probes run notebook-probe --restart=Never --dry-run=server \
    --image="$1" --overrides="${POD_SPEC/IMAGE/$1}" 2>&1 | fold -s -w 140
  return 0
}
echo "# image from a registry outside the allowlist:"
try_pod docker.io/library/busybox:1.36
echo
echo "# tenant image Kyverno cannot verify (the tag does not exist, so there is no signed digest):"
try_pod ghcr.io/brunobml/orders-processor:not-signed

**Reading the output.** Two different admission layers answered:

1. **`ValidatingAdmissionPolicy 'tenant-image-registry-allowlist' … denied`**: a built-in Kubernetes
   policy (CEL, no webhook) that allows only images from `ghcr.io/brunobml/` in tenant namespaces.
2. **`admission webhook "ivpol.mutate.kyverno…" denied`**: Kyverno's image policy tried to resolve the
   tag to a digest to verify its signature; the tag does not exist, so it fails closed.

Read a denial from the start: it names the layer, then the policy, then the reason. Nothing was
created (`--dry-run=server`), but the denials count in the admission metrics of section 7.

## 7. Observability: Prometheus and Loki

The spokes' Prometheus agents and Alloy send metrics and logs to the hub. Grafana
(http://grafana.localhost) is the place to look; these helpers query the same data from the
terminal by running `wget` inside the hub Prometheus pod, the same way the smoke tests do.

In [ ]:
promq() {
  kubectl --context "$HUB" -n monitoring exec deploy/prometheus-server -c prometheus-server -- \
    wget -qO- "http://localhost:9090/api/v1/query?query=$(jq -rn --arg q "$1" '$q|@uri')"
}
lokiq() {  # LogQL query over the last hour, newest first
  local q start
  q=$(jq -rn --arg q "$1" '$q|@uri')
  start=$(( $(date +%s) - 3600 ))000000000
  kubectl --context "$HUB" -n monitoring exec deploy/prometheus-server -c prometheus-server -- \
    wget -qO- "http://loki:3100/loki/api/v1/query_range?query=${q}&start=${start}&limit=${2:-10}"
}
echo "✔ promq and lokiq defined"

Targets per cluster that are up or down:

In [ ]:
promq 'sum by (cluster) (up)' | jq -r '.data.result[] | "up\t\(.metric.cluster // "(no cluster label)")\t\(.value[1])"'
promq 'up == 0' | jq -r '.data.result[] | "DOWN\t\(.metric.cluster // "hub")\t\(.metric.job)\t\(.metric.instance)"'

**Reading the output.** How many scrape targets are up, per cluster, then
every target that is down.

- **`(no cluster label)`** are the targets of the hub's own Prometheus; the spoke agents add
  `cluster="spoke-…"` before they send, and a few hub targets set `cluster="hub"` themselves.
- **A `DOWN` line** means no fresh metrics from that target, so dashboards and alerts built on it go
  quiet. Check its Pod first: `kubectl get pods -n <ns>` on that cluster.

Kyverno admission requests per spoke and decision over the last hour:

In [ ]:
promq 'sum by (cluster, request_allowed) (increase(kyverno_admission_requests_total[1h]))' \
  | jq -r '.data.result[] | "\(.metric.cluster)\tallowed=\(.metric.request_allowed)\t\(.value[1] | tonumber | round)"' | column -t -s $'\t'

**Reading the output.** Admission requests Kyverno decided in the last hour,
per spoke. `allowed=false` counts denials; the dry-runs of section 6 add to it.

- **A jump in denials after a policy change:** the policy blocks more than intended.
- **No requests at all:** the webhook is not being called (Kyverno down, or its webhook configuration
  gone), so nothing is checked.

Grafana's *Kyverno — Daily Operations* dashboard graphs the same metric.

Which labels Loki knows, and the last log lines of one namespace containing `error`:

In [ ]:
kubectl --context "$HUB" -n monitoring exec deploy/prometheus-server -c prometheus-server -- \
  wget -qO- http://loki:3100/loki/api/v1/labels | jq -M -c .data
lokiq '{namespace="kyverno"} |= "error"' 5 \
  | jq -r '.data.result[] | .stream.pod as $p | .values[] | "\($p)  \(.[1] | .[0:160])"' \
  | sed 's/\x1b\[[0-9;]*m//g'   # drop the colour codes some apps write into their logs
echo "(end of log lines)"

**Reading the output.** First, the labels a LogQL query can filter on
(`{namespace="…", pod="…"}`). Then the newest matching lines: pod, then the log line (cut at 160
characters). Here the Kyverno admission controller logs `failed to update digest` for each image it
could not resolve, such as the unsigned-image dry-run in section 6. `|= "error"` is a case-sensitive
substring filter; `|~ "(?i)error"` ignores case. Grafana → Explore → Loki runs the same queries.

## 8. Troubleshooting one-liners

Pods that are not Running or Completed, on all clusters:

In [ ]:
for c in "$HUB" "${SPOKES[@]}"; do
  echo "== $c"
  kubectl --context "$c" get pods -A --field-selector=status.phase!=Running,status.phase!=Succeeded --no-headers 2>&1
done

**Reading the output.** `No resources found` on
every cluster is the good result. Otherwise the STATUS column says where to look:

- **`Pending`**: not scheduled (not enough CPU or memory, a missing volume): `kubectl describe pod`,
  Events at the bottom.
- **`Failed`**: the Pod ended with an error; Jobs leave these behind.
- **`Unknown`**: the node stopped reporting.

A crash-looping Pod stays `Running` between restarts, so it does not appear here: see the restarts
cell.

Warning events, newest last:

In [ ]:
kubectl --context "$NONPROD" get events -A --field-selector type=Warning --sort-by=.metadata.creationTimestamp \
  -o custom-columns='TIME:.metadata.creationTimestamp,NS:.metadata.namespace,OBJECT:.involvedObject.kind,NAME:.involvedObject.name,REASON:.reason' | tail -15

**Reading the output.** Warning events, newest last: time, namespace,
object and reason.

- **`PolicyViolation`**: Kyverno reports each audit failure as an event on the failing object and on
  the policy (the `ValidatingPolicy` lines in `default`).
- **Other reasons to know:** `BackOff` (a container keeps restarting), `FailedScheduling`,
  `FailedMount`, `Unhealthy` (a probe failed).

Events expire after about an hour. The message is in `kubectl describe <kind> <name> -n <ns>`.

Pods with restarts, most first, and the biggest memory users:

In [ ]:
kubectl --context "$NONPROD" get pods -A -o json | jq -r '
  .items[] | {ns: .metadata.namespace, pod: .metadata.name, r: ([.status.containerStatuses[]?.restartCount] | add // 0)}
  | select(.r > 0) | "\(.r)\t\(.ns)/\(.pod)"' | sort -rn | head -10
echo
kubectl --context "$NONPROD" top pods -A --sort-by=memory | head -8

**Reading the output.** Restart counts, highest first, then the Pods using the most memory.

- **Most restarts here come from host reboots** and `make stop` / `make start`: restarting a node
  restarts its containers.
- **A count that keeps growing while the lab runs** means a crash. `kubectl logs --previous` shows
  the last crashed run; `kubectl describe pod` shows `Last State: Terminated` with a reason, such as
  `OOMKilled` (the memory limit is too low).
- **`MEMORY`** is the working set reported by metrics-server (part of k3s); compare it with the
  Pod's memory limit.

Every container image in a namespace (handy for the image policies):

In [ ]:
kubectl --context "$NONPROD" -n orders-dev get pods \
  -o jsonpath='{range .items[*]}{.metadata.name}{"\t"}{range .spec.containers[*]}{.image}{" "}{end}{"\n"}{end}'

**Reading the output.** Pod name, then each container image. Tenant images carry a
tag and a digest (`:v1.5.0@sha256:…`). The digest pins the exact image: it is what
`tenant-images-signed` verifies the signature of and what `require-image-digest` asks for. A tag alone
could point to a different image tomorrow.

API server health of each cluster:

In [ ]:
for c in "$HUB" "${SPOKES[@]}"; do
  printf '%-20s ' "$c"; kubectl --context "$c" get --raw='/readyz' 2>&1; echo
done

**Reading the output.** `ok` from each API server's `/readyz` endpoint. Anything else, or a
timeout, means kubectl cannot work with that cluster: start with `docker ps` for its server container.
`kubectl get --raw='/readyz?verbose'` lists the individual checks.

**Stale load balancer.** After a host restart a spoke's k3d load balancer can keep old node IPs, and
every second request then returns 404. Six requests in a row must all return 200 (smoke Gate 9b).
The fix is in *Changes go through Git* below (`make start` does it).

In [ ]:
for pair in orders-dev:8081 orders-test:8081 orders-prod:8082; do
  IFS=: read -r ns port <<<"$pair"
  codes=""
  for i in 1 2 3 4 5 6; do codes+="$(curl -s -m 5 -o /dev/null -w '%{http_code}' "http://${ns}.localhost:${port}/") "; done
  echo "${ns}: ${codes}"
done

**Reading the output.** Six `200`s per app is healthy.

- **`200 404 200 404`**: the stale load balancer described above.
- **All `404`**: the app or its Ingress is missing.
- **`000`**: nothing answered: the spoke is not running, or the port is wrong (nonprod apps on 8081,
  prod on 8082).

## Changes go through Git

These commands change the lab. They are here so you recognize them, not to run from this
notebook. Prefer the Git route; use the direct one only in a drill or an incident.

| Goal | Git-first way | Direct command (drills and incidents) |
| --- | --- | --- |
| Change a manifest or values | edit, `git commit`, `git push`; Argo CD syncs | `kubectl apply` (self-heal reverts it) |
| Sync now instead of waiting | push, then refresh in the UI | `argocd app sync <app>` |
| Promote blueprints to prod | tag platform-catalog, edit `clusters/blueprint-revisions.env`, `make promote-blueprints` | — |
| Restart a workload | change its template in Git (a checksum annotation) | `kubectl rollout restart deploy/<name> -n <ns>` |
| Recover Moto after a restart | `make moto-restart` | — |
| Fix a stale spoke load balancer | `make start` | `docker exec k3d-<spoke>-serverlb nginx -s reload` |
| Renew credentials and tokens | `make maintain` | — |

Passwords: `make password` in a terminal, not in a cell (outputs end up in files).

## Cleanup

Removes the temporary Argo CD config of section 2.

In [ ]:
rm -f "${ARGOCD_CFG:-}" && unset ARGOCD_CFG A && echo "✔ temporary Argo CD config removed"